In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from sympy import lambdify
import sympy as sp

# 1. Point Python to your exact SRC directory
sys.path.append("/Users/adarshmac/simulations/bbh_run1/ANALYTICAL_CODES/SRC")

# 2. Import the engine AND EVERY metric available in your library
from Testsymbolicengine import (
    calculate_automated_fields, get_schwarzschild_spherical, get_schwarzschild_isotropic,
    get_reissner_nordstrom_spherical, get_reissner_nordstrom_isotropic,
    get_flrw_cartesian, get_bardeen_spherical, get_kerr_boyer_lindquist,
    get_kerr_quasi_isotropic_spherical, get_kerr_schild_cartesian
)

# 3. Load the geometry 
print("Loading Metric Geometry...")
metric_data = get_kerr_schild_cartesian() # Test FLRW!

# 4. Execute the engine
print("Executing Tensor Calculus Engine...")
results = calculate_automated_fields(metric_data)

# 5. Extract matrices
E_hat = results['E_hat']
rho_hat = results['rho_hat']
q_hat = results['q_hat']

# 6. UNIVERSAL SYMBOL EXTRACTION
print("Extracting variables...")
syms = results['symbols']

M = syms.get('M')
x = syms.get('x')
y = syms.get('y')
z = syms.get('z')

# Safely catch both standard 'r' and isotropic 'r_bar'
r = syms.get('r') if syms.get('r') is not None else syms.get('r_bar')
theta = syms.get('theta')
phi = syms.get('phi')

a = syms.get('a')    
Q = syms.get('Q')    
Qm = syms.get('Qm')  

print("Ready for plotting!")

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from sympy import lambdify
import sympy as sp

# 1. Point Python to your exact SRC directory
sys.path.append("/Users/adarshmac/simulations/bbh_run1/ANALYTICAL_CODES/SRC")

# 2. Import the engine AND EVERY metric available in your library
from Testsymbolicengine import (
    calculate_automated_fields, get_schwarzschild_spherical, get_schwarzschild_isotropic,
    get_reissner_nordstrom_spherical, get_reissner_nordstrom_isotropic,
    get_flrw_cartesian, get_bardeen_spherical, get_kerr_boyer_lindquist,
    get_kerr_quasi_isotropic_spherical, get_kerr_schild_cartesian
)

# 3. Load the geometry 
print("Loading Metric Geometry...")
metric_data = get_kerr_schild_cartesian() # Load Kerr-Schild!

# 4. Execute the engine
print("Executing Tensor Calculus Engine...")
results = calculate_automated_fields(metric_data)

# 5. Extract ALL matrices (This is the critical update!)
E_hat = results['E_hat']
D_hat = results['D_hat']
B_hat = results['B_hat']
H_hat = results['H_hat']
rho_hat = results['rho_hat']
q_hat = results['q_hat']
s_hat = results['s_hat']
j_hat = results['j_hat']

# 6. UNIVERSAL SYMBOL EXTRACTION
print("Extracting variables...")
syms = results['symbols']

M = syms.get('M')
x = syms.get('x')
y = syms.get('y')
z = syms.get('z')

# Safely catch both standard 'r' and isotropic 'r_bar'
r = syms.get('r') if syms.get('r') is not None else syms.get('r_bar')
theta = syms.get('theta')
phi = syms.get('phi')

a = syms.get('a')    
Q = syms.get('Q')    
Qm = syms.get('Qm')  

print("Ready for plotting!")

In [2]:
#With Simplify

def plot_analytical_fields_cartesian(E_hat, q_hat, rho_hat, x_sym, y_sym, z_sym, param_dict, horizon_radius):
    print("Lambdifying exact symbolic expressions into NumPy functions...")
    
    # Substitute ALL physics parameters at once
    Ex_expr = sp.simplify(E_hat[0, 1].subs(param_dict))
    Ez_expr = sp.simplify(E_hat[0, 3].subs(param_dict))
    q0_expr = sp.simplify(q_hat[0].subs(param_dict))
    rho0_expr = sp.simplify(rho_hat[0].subs(param_dict))
    
    Ex_func = lambdify((x_sym, y_sym, z_sym), Ex_expr, "numpy")
    Ez_func = lambdify((x_sym, y_sym, z_sym), Ez_expr, "numpy")
    q0_func = lambdify((x_sym, y_sym, z_sym), q0_expr, "numpy")
    rho0_func = lambdify((x_sym, y_sym, z_sym), rho0_expr, "numpy")
    
    print("Evaluating analytical fields over the 2D Cartesian grid...")
    grid_lim = 8.0
    resolution = 300
    x_vals = np.linspace(-grid_lim, grid_lim, resolution)
    z_vals = np.linspace(-grid_lim, grid_lim, resolution)
    X, Z = np.meshgrid(x_vals, z_vals)
    Y = np.zeros_like(X) # Evaluate on the equatorial slice y=0
    
    R = np.sqrt(X**2 + Z**2)
    
    mask = R > (1.02 * horizon_radius)
    X_safe = np.where(mask, X, 1.02 * horizon_radius)
    Y_safe = np.where(mask, Y, 0.0)
    Z_safe = np.where(mask, Z, 0.0)
    
    Ex_num = np.zeros_like(R)
    Ez_num = np.zeros_like(R)
    q0_num = np.zeros_like(R)
    rho0_num = np.zeros_like(R)
    
    Ex_num[mask] = Ex_func(X_safe[mask], Y_safe[mask], Z_safe[mask])
    Ez_num[mask] = Ez_func(X_safe[mask], Y_safe[mask], Z_safe[mask])
    q0_num[mask] = q0_func(X_safe[mask], Y_safe[mask], Z_safe[mask])
    rho0_num[mask] = rho0_func(X_safe[mask], Y_safe[mask], Z_safe[mask])
    
    Ex_num[~mask] = np.nan
    Ez_num[~mask] = np.nan
    q0_num[~mask] = np.nan
    rho0_num[~mask] = np.nan

    print("Rendering Plots...")
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    
    # Panel 1: Physical Density
    ax1 = axes[0]
    rho_max = np.nanmax(np.abs(rho0_num))
    if np.isnan(rho_max) or rho_max == 0: rho_max = 1.0
    cmap1 = ax1.pcolormesh(X, Z, rho0_num, shading='auto', cmap='RdBu_r', vmin=-rho_max, vmax=rho_max)
    fig.colorbar(cmap1, ax=ax1, label=r"Gravitational Energy Density ($\rho_{\hat{0}}$)")
    ax1.streamplot(x_vals, z_vals, Ex_num, Ez_num, color='black', density=1.5, linewidth=0.8)
    ax1.add_patch(plt.Circle((0, 0), horizon_radius, color='black', zorder=10))
    ax1.set_aspect('equal')
    ax1.set_title(r"True Energy Density: $\rho_{\hat{0}}$ (Cartesian)", fontsize=14, fontweight='bold')
    ax1.set_xlabel("X (M)", fontsize=12)
    ax1.set_ylabel("Z (M)", fontsize=12)

    # Panel 2: Macroscopic Charge
    ax2 = axes[1]
    q_max = np.nanmax(np.abs(q0_num))
    if np.isnan(q_max) or q_max == 0: q_max = 1.0 
    cmap2 = ax2.pcolormesh(X, Z, q0_num, shading='auto', cmap='RdBu_r', vmin=-q_max, vmax=q_max)
    fig.colorbar(cmap2, ax=ax2, label=r"Spacetime Energy Charge ($q_{\hat{0}}$)")
    ax2.streamplot(x_vals, z_vals, Ex_num, Ez_num, color='black', density=1.5, linewidth=0.8)
    ax2.add_patch(plt.Circle((0, 0), horizon_radius, color='black', zorder=10))
    ax2.set_aspect('equal')
    ax2.set_title(r"Macroscopic Charge: $q_{\hat{0}}$", fontsize=14, fontweight='bold')
    ax2.set_xlabel("X (M)", fontsize=12)

    plt.tight_layout()
    plt.show()

def plot_analytical_fields_spherical(E_hat, q_hat, rho_hat, r_sym, theta_sym, param_dict, horizon_radius):
    print("Lambdifying exact symbolic expressions into NumPy functions...")
    
    E_r_expr = sp.simplify(E_hat[0, 1].subs(param_dict))
    E_theta_expr = sp.simplify(E_hat[0, 2].subs(param_dict))
    q0_expr = sp.simplify(q_hat[0].subs(param_dict))
    rho0_expr = sp.simplify(rho_hat[0].subs(param_dict))
    
    E_r_func = lambdify((r_sym, theta_sym), E_r_expr, "numpy")
    E_theta_func = lambdify((r_sym, theta_sym), E_theta_expr, "numpy")
    q0_func = lambdify((r_sym, theta_sym), q0_expr, "numpy")
    rho0_func = lambdify((r_sym, theta_sym), rho0_expr, "numpy")
    
    print("Evaluating analytical fields over the 2D Spherical grid...")
    grid_lim = 8.0
    resolution = 300
    x_vals = np.linspace(-grid_lim, grid_lim, resolution)
    z_vals = np.linspace(-grid_lim, grid_lim, resolution)
    X, Z = np.meshgrid(x_vals, z_vals)
    
    R = np.sqrt(X**2 + Z**2)
    R = np.where(R < 1e-5, 1e-5, R) 
    Theta = np.arccos(Z / R)
    
    mask = R > (1.02 * horizon_radius)
    R_safe = np.where(mask, R, 1.02 * horizon_radius)
    Theta_safe = np.where(mask, Theta, np.pi/2)

    E_r_num = np.zeros_like(R)
    E_theta_num = np.zeros_like(R)
    q0_num = np.zeros_like(R)
    rho0_num = np.zeros_like(R)
    
    E_r_num[mask] = E_r_func(R_safe[mask], Theta_safe[mask])
    E_theta_num[mask] = E_theta_func(R_safe[mask], Theta_safe[mask])
    q0_num[mask] = q0_func(R_safe[mask], Theta_safe[mask])
    rho0_num[mask] = rho0_func(R_safe[mask], Theta_safe[mask])
    
    sign_x = np.sign(X)
    sign_x[sign_x == 0] = 1.0 
    
    Ex = (E_r_num * np.sin(Theta) + E_theta_num * np.cos(Theta)) * sign_x
    Ez = E_r_num * np.cos(Theta) - E_theta_num * np.sin(Theta)
    
    Ex[~mask] = np.nan
    Ez[~mask] = np.nan
    q0_num[~mask] = np.nan
    rho0_num[~mask] = np.nan

    print("Rendering Plots...")
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    
    ax1 = axes[0]
    rho_max = np.nanmax(np.abs(rho0_num))
    if np.isnan(rho_max) or rho_max == 0: rho_max = 1.0
    cmap1 = ax1.pcolormesh(X, Z, rho0_num, shading='auto', cmap='RdBu_r', vmin=-rho_max, vmax=rho_max)
    fig.colorbar(cmap1, ax=ax1, label=r"Gravitational Energy Density ($\rho_{\hat{0}}$)")
    ax1.streamplot(x_vals, z_vals, Ex, Ez, color='black', density=1.5, linewidth=0.8)
    ax1.add_patch(plt.Circle((0, 0), horizon_radius, color='black', zorder=10))
    ax1.set_aspect('equal')
    ax1.set_title(r"True Energy Density: $\rho_{\hat{0}}$ (Spherical)", fontsize=14, fontweight='bold')
    ax1.set_xlabel("X (M)", fontsize=12)
    ax1.set_ylabel("Z (M)", fontsize=12)

    ax2 = axes[1]
    q_max = np.nanmax(np.abs(q0_num))
    if np.isnan(q_max) or q_max == 0: q_max = 1.0 
    cmap2 = ax2.pcolormesh(X, Z, q0_num, shading='auto', cmap='RdBu_r', vmin=-q_max, vmax=q_max)
    fig.colorbar(cmap2, ax=ax2, label=r"Spacetime Energy Charge ($q_{\hat{0}}$)")
    ax2.streamplot(x_vals, z_vals, Ex, Ez, color='black', density=1.5, linewidth=0.8)
    ax2.add_patch(plt.Circle((0, 0), horizon_radius, color='black', zorder=10))
    ax2.set_aspect('equal')
    ax2.set_title(r"Macroscopic Charge: $q_{\hat{0}}$", fontsize=14, fontweight='bold')
    ax2.set_xlabel("X (M)", fontsize=12)

    plt.tight_layout()
    plt.show()

In [2]:
##Without Simplify 

def plot_analytical_fields_cartesian(E_hat, q_hat, rho_hat, x_sym, y_sym, z_sym, param_dict, horizon_radius):
    print("Lambdifying exact symbolic expressions into NumPy functions...")
    
    # ABSOLUTELY NO sp.simplify() ALLOWED HERE!
    Ex_expr = E_hat[0, 1].subs(param_dict)
    Ez_expr = E_hat[0, 3].subs(param_dict)
    q0_expr = q_hat[0].subs(param_dict)
    rho0_expr = rho_hat[0].subs(param_dict)
    
    Ex_func = lambdify((x_sym, y_sym, z_sym), Ex_expr, "numpy")
    Ez_func = lambdify((x_sym, y_sym, z_sym), Ez_expr, "numpy")
    q0_func = lambdify((x_sym, y_sym, z_sym), q0_expr, "numpy")
    rho0_func = lambdify((x_sym, y_sym, z_sym), rho0_expr, "numpy")
    
    print("Evaluating analytical fields over the 2D Cartesian grid...")
    grid_lim = 8.0
    resolution = 300
    x_vals = np.linspace(-grid_lim, grid_lim, resolution)
    z_vals = np.linspace(-grid_lim, grid_lim, resolution)
    X, Z = np.meshgrid(x_vals, z_vals)
    Y = np.zeros_like(X) # Evaluate on the equatorial slice y=0
    
    R = np.sqrt(X**2 + Z**2)
    
    mask = R > (1.02 * horizon_radius)
    X_safe = np.where(mask, X, 1.02 * horizon_radius)
    Y_safe = np.where(mask, Y, 0.0)
    Z_safe = np.where(mask, Z, 0.0)
    
    Ex_num = np.zeros_like(R)
    Ez_num = np.zeros_like(R)
    q0_num = np.zeros_like(R)
    rho0_num = np.zeros_like(R)
    
    Ex_num[mask] = Ex_func(X_safe[mask], Y_safe[mask], Z_safe[mask])
    Ez_num[mask] = Ez_func(X_safe[mask], Y_safe[mask], Z_safe[mask])
    q0_num[mask] = q0_func(X_safe[mask], Y_safe[mask], Z_safe[mask])
    rho0_num[mask] = rho0_func(X_safe[mask], Y_safe[mask], Z_safe[mask])
    
    Ex_num[~mask] = np.nan
    Ez_num[~mask] = np.nan
    q0_num[~mask] = np.nan
    rho0_num[~mask] = np.nan

    print("Rendering Plots...")
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    
    ax1 = axes[0]
    rho_max = np.nanmax(np.abs(rho0_num))
    if np.isnan(rho_max) or rho_max == 0: rho_max = 1.0
    cmap1 = ax1.pcolormesh(X, Z, rho0_num, shading='auto', cmap='RdBu_r', vmin=-rho_max, vmax=rho_max)
    fig.colorbar(cmap1, ax=ax1, label=r"Gravitational Energy Density ($\rho_{\hat{0}}$)")
    ax1.streamplot(x_vals, z_vals, Ex_num, Ez_num, color='black', density=1.5, linewidth=0.8)
    ax1.add_patch(plt.Circle((0, 0), horizon_radius, color='black', zorder=10))
    ax1.set_aspect('equal')
    ax1.set_title(r"True Energy Density: $\rho_{\hat{0}}$ (Cartesian)", fontsize=14, fontweight='bold')
    ax1.set_xlabel("X (M)", fontsize=12)
    ax1.set_ylabel("Z (M)", fontsize=12)

    ax2 = axes[1]
    q_max = np.nanmax(np.abs(q0_num))
    if np.isnan(q_max) or q_max == 0: q_max = 1.0 
    cmap2 = ax2.pcolormesh(X, Z, q0_num, shading='auto', cmap='RdBu_r', vmin=-q_max, vmax=q_max)
    fig.colorbar(cmap2, ax=ax2, label=r"Spacetime Energy Charge ($q_{\hat{0}}$)")
    ax2.streamplot(x_vals, z_vals, Ex_num, Ez_num, color='black', density=1.5, linewidth=0.8)
    ax2.add_patch(plt.Circle((0, 0), horizon_radius, color='black', zorder=10))
    ax2.set_aspect('equal')
    ax2.set_title(r"Macroscopic Charge: $q_{\hat{0}}$", fontsize=14, fontweight='bold')
    ax2.set_xlabel("X (M)", fontsize=12)

    plt.tight_layout()
    plt.show()

def plot_analytical_fields_spherical(E_hat, q_hat, rho_hat, r_sym, theta_sym, param_dict, horizon_radius):
    print("Lambdifying exact symbolic expressions into NumPy functions...")
    
    # ABSOLUTELY NO sp.simplify() ALLOWED HERE!
    E_r_expr = E_hat[0, 1].subs(param_dict)
    E_theta_expr = E_hat[0, 2].subs(param_dict)
    q0_expr = q_hat[0].subs(param_dict)
    rho0_expr = rho_hat[0].subs(param_dict)
    
    E_r_func = lambdify((r_sym, theta_sym), E_r_expr, "numpy")
    E_theta_func = lambdify((r_sym, theta_sym), E_theta_expr, "numpy")
    q0_func = lambdify((r_sym, theta_sym), q0_expr, "numpy")
    rho0_func = lambdify((r_sym, theta_sym), rho0_expr, "numpy")
    
    print("Evaluating analytical fields over the 2D Spherical grid...")
    grid_lim = 8.0
    resolution = 300
    x_vals = np.linspace(-grid_lim, grid_lim, resolution)
    z_vals = np.linspace(-grid_lim, grid_lim, resolution)
    X, Z = np.meshgrid(x_vals, z_vals)
    
    R = np.sqrt(X**2 + Z**2)
    R = np.where(R < 1e-5, 1e-5, R) 
    Theta = np.arccos(Z / R)
    
    mask = R > (1.02 * horizon_radius)
    R_safe = np.where(mask, R, 1.02 * horizon_radius)
    Theta_safe = np.where(mask, Theta, np.pi/2)

    E_r_num = np.zeros_like(R)
    E_theta_num = np.zeros_like(R)
    q0_num = np.zeros_like(R)
    rho0_num = np.zeros_like(R)
    
    E_r_num[mask] = E_r_func(R_safe[mask], Theta_safe[mask])
    E_theta_num[mask] = E_theta_func(R_safe[mask], Theta_safe[mask])
    q0_num[mask] = q0_func(R_safe[mask], Theta_safe[mask])
    rho0_num[mask] = rho0_func(R_safe[mask], Theta_safe[mask])
    
    sign_x = np.sign(X)
    sign_x[sign_x == 0] = 1.0 
    
    Ex = (E_r_num * np.sin(Theta) + E_theta_num * np.cos(Theta)) * sign_x
    Ez = E_r_num * np.cos(Theta) - E_theta_num * np.sin(Theta)
    
    Ex[~mask] = np.nan
    Ez[~mask] = np.nan
    q0_num[~mask] = np.nan
    rho0_num[~mask] = np.nan

    print("Rendering Plots...")
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    
    ax1 = axes[0]
    rho_max = np.nanmax(np.abs(rho0_num))
    if np.isnan(rho_max) or rho_max == 0: rho_max = 1.0
    cmap1 = ax1.pcolormesh(X, Z, rho0_num, shading='auto', cmap='RdBu_r', vmin=-rho_max, vmax=rho_max)
    fig.colorbar(cmap1, ax=ax1, label=r"Gravitational Energy Density ($\rho_{\hat{0}}$)")
    ax1.streamplot(x_vals, z_vals, Ex, Ez, color='black', density=1.5, linewidth=0.8)
    ax1.add_patch(plt.Circle((0, 0), horizon_radius, color='black', zorder=10))
    ax1.set_aspect('equal')
    ax1.set_title(r"True Energy Density: $\rho_{\hat{0}}$ (Spherical)", fontsize=14, fontweight='bold')
    ax1.set_xlabel("X (M)", fontsize=12)
    ax1.set_ylabel("Z (M)", fontsize=12)

    ax2 = axes[1]
    q_max = np.nanmax(np.abs(q0_num))
    if np.isnan(q_max) or q_max == 0: q_max = 1.0 
    cmap2 = ax2.pcolormesh(X, Z, q0_num, shading='auto', cmap='RdBu_r', vmin=-q_max, vmax=q_max)
    fig.colorbar(cmap2, ax=ax2, label=r"Spacetime Energy Charge ($q_{\hat{0}}$)")
    ax2.streamplot(x_vals, z_vals, Ex, Ez, color='black', density=1.5, linewidth=0.8)
    ax2.add_patch(plt.Circle((0, 0), horizon_radius, color='black', zorder=10))
    ax2.set_aspect('equal')
    ax2.set_title(r"Macroscopic Charge: $q_{\hat{0}}$", fontsize=14, fontweight='bold')
    ax2.set_xlabel("X (M)", fontsize=12)

    plt.tight_layout()
    plt.show()

In [5]:
#Simplified Lamdify implementation for faster plotting without sacrificing correctness (CSE Activated, but no sp.simplify() allowed!) and FLRW implementation is optimised.
def plot_analytical_fields_cartesian(E_hat, q_hat, rho_hat, x_sym, y_sym, z_sym, param_dict, horizon_radius):
    print("Pre-evaluating on the Y=0 plane to collapse massive expressions...")
    
    full_subs = param_dict.copy()
    full_subs[y_sym] = 0.0 
    
    # simultaneous=True PREVENTS the derivative from evaluating to zero!
    Ex_expr = E_hat[0, 1].subs(full_subs, simultaneous=True).doit()
    Ez_expr = E_hat[0, 3].subs(full_subs, simultaneous=True).doit()
    q0_expr = q_hat[0].subs(full_subs, simultaneous=True).doit()
    rho0_expr = rho_hat[0].subs(full_subs, simultaneous=True).doit()
    
    print("Compiling AST to C-code (CSE Activated)...")
    Ex_func = lambdify((x_sym, z_sym), Ex_expr, "numpy", cse=True)
    Ez_func = lambdify((x_sym, z_sym), Ez_expr, "numpy", cse=True)
    q0_func = lambdify((x_sym, z_sym), q0_expr, "numpy", cse=True)
    rho0_func = lambdify((x_sym, z_sym), rho0_expr, "numpy", cse=True)
    
    print("Evaluating over 2D grid...")
    grid_lim = 8.0
    resolution = 300
    x_vals = np.linspace(-grid_lim, grid_lim, resolution)
    z_vals = np.linspace(-grid_lim, grid_lim, resolution)
    X, Z = np.meshgrid(x_vals, z_vals)
    R = np.sqrt(X**2 + Z**2)
    
    hr = horizon_radius if horizon_radius is not None else 0.0
    mask = R > (1.02 * hr) if hr > 0 else np.ones_like(R, dtype=bool)
    
    X_safe = np.where(mask, X, 1.02 * hr)
    Z_safe = np.where(mask, Z, 0.0)
    
    Ex_num = np.zeros_like(R)
    Ez_num = np.zeros_like(R)
    q0_num = np.zeros_like(R)
    rho0_num = np.zeros_like(R)
    
    Ex_num[mask] = Ex_func(X_safe[mask], Z_safe[mask])
    Ez_num[mask] = Ez_func(X_safe[mask], Z_safe[mask])
    q0_num[mask] = q0_func(X_safe[mask], Z_safe[mask])
    rho0_num[mask] = rho0_func(X_safe[mask], Z_safe[mask])
    
    Ex_num[~mask] = np.nan
    Ez_num[~mask] = np.nan
    q0_num[~mask] = np.nan
    rho0_num[~mask] = np.nan

    print("Rendering Plots...")
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    
    ax1 = axes[0]
    rho_max = np.nanmax(np.abs(rho0_num))
    if np.isnan(rho_max) or rho_max == 0: rho_max = 1.0
    cmap1 = ax1.pcolormesh(X, Z, rho0_num, shading='auto', cmap='RdBu_r', vmin=-rho_max, vmax=rho_max)
    fig.colorbar(cmap1, ax=ax1, label=r"Gravitational Energy Density ($\rho_{\hat{0}}$)")
    ax1.streamplot(x_vals, z_vals, Ex_num, Ez_num, color='black', density=1.5, linewidth=0.8)
    if hr > 0: ax1.add_patch(plt.Circle((0, 0), hr, color='black', zorder=10))
    ax1.set_aspect('equal')
    ax1.set_title(r"True Energy Density: $\rho_{\hat{0}}$ (Cartesian)", fontsize=14, fontweight='bold')

    ax2 = axes[1]
    q_max = np.nanmax(np.abs(q0_num))
    if np.isnan(q_max) or q_max == 0: q_max = 1.0 
    cmap2 = ax2.pcolormesh(X, Z, q0_num, shading='auto', cmap='RdBu_r', vmin=-q_max, vmax=q_max)
    fig.colorbar(cmap2, ax=ax2, label=r"Spacetime Energy Charge ($q_{\hat{0}}$)")
    ax2.streamplot(x_vals, z_vals, Ex_num, Ez_num, color='black', density=1.5, linewidth=0.8)
    if hr > 0: ax2.add_patch(plt.Circle((0, 0), hr, color='black', zorder=10))
    ax2.set_aspect('equal')
    ax2.set_title(r"Macroscopic Charge: $q_{\hat{0}}$", fontsize=14, fontweight='bold')

    plt.tight_layout()
    plt.show()

def plot_analytical_fields_spherical(E_hat, q_hat, rho_hat, r_sym, theta_sym, param_dict, horizon_radius):
    print("Compiling AST to C-code (CSE Activated)...")
    
    # simultaneous=True added here as well
    E_r_expr = E_hat[0, 1].subs(param_dict, simultaneous=True).doit()
    E_theta_expr = E_hat[0, 2].subs(param_dict, simultaneous=True).doit()
    q0_expr = q_hat[0].subs(param_dict, simultaneous=True).doit()
    rho0_expr = rho_hat[0].subs(param_dict, simultaneous=True).doit()
    
    E_r_func = lambdify((r_sym, theta_sym), E_r_expr, "numpy", cse=True)
    E_theta_func = lambdify((r_sym, theta_sym), E_theta_expr, "numpy", cse=True)
    q0_func = lambdify((r_sym, theta_sym), q0_expr, "numpy", cse=True)
    rho0_func = lambdify((r_sym, theta_sym), rho0_expr, "numpy", cse=True)
    
    print("Evaluating over 2D Spherical grid...")
    grid_lim = 8.0
    resolution = 300
    x_vals = np.linspace(-grid_lim, grid_lim, resolution)
    z_vals = np.linspace(-grid_lim, grid_lim, resolution)
    X, Z = np.meshgrid(x_vals, z_vals)
    
    R = np.sqrt(X**2 + Z**2)
    R = np.where(R < 1e-5, 1e-5, R) 
    Theta = np.arccos(Z / R)
    
    mask = R > (1.02 * horizon_radius)
    R_safe = np.where(mask, R, 1.02 * horizon_radius)
    Theta_safe = np.where(mask, Theta, np.pi/2)

    E_r_num = np.zeros_like(R)
    E_theta_num = np.zeros_like(R)
    q0_num = np.zeros_like(R)
    rho0_num = np.zeros_like(R)
    
    E_r_num[mask] = E_r_func(R_safe[mask], Theta_safe[mask])
    E_theta_num[mask] = E_theta_func(R_safe[mask], Theta_safe[mask])
    q0_num[mask] = q0_func(R_safe[mask], Theta_safe[mask])
    rho0_num[mask] = rho0_func(R_safe[mask], Theta_safe[mask])
    
    sign_x = np.sign(X)
    sign_x[sign_x == 0] = 1.0 
    
    Ex = (E_r_num * np.sin(Theta) + E_theta_num * np.cos(Theta)) * sign_x
    Ez = E_r_num * np.cos(Theta) - E_theta_num * np.sin(Theta)
    
    Ex[~mask] = np.nan
    Ez[~mask] = np.nan
    q0_num[~mask] = np.nan
    rho0_num[~mask] = np.nan

    print("Rendering Plots...")
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    
    ax1 = axes[0]
    rho_max = np.nanmax(np.abs(rho0_num))
    if np.isnan(rho_max) or rho_max == 0: rho_max = 1.0
    cmap1 = ax1.pcolormesh(X, Z, rho0_num, shading='auto', cmap='RdBu_r', vmin=-rho_max, vmax=rho_max)
    fig.colorbar(cmap1, ax=ax1, label=r"Gravitational Energy Density ($\rho_{\hat{0}}$)")
    ax1.streamplot(x_vals, z_vals, Ex, Ez, color='black', density=1.5, linewidth=0.8)
    ax1.add_patch(plt.Circle((0, 0), horizon_radius, color='black', zorder=10))
    ax1.set_aspect('equal')
    ax1.set_title(r"True Energy Density: $\rho_{\hat{0}}$ (Spherical)", fontsize=14, fontweight='bold')

    ax2 = axes[1]
    q_max = np.nanmax(np.abs(q0_num))
    if np.isnan(q_max) or q_max == 0: q_max = 1.0 
    cmap2 = ax2.pcolormesh(X, Z, q0_num, shading='auto', cmap='RdBu_r', vmin=-q_max, vmax=q_max)
    fig.colorbar(cmap2, ax=ax2, label=r"Spacetime Energy Charge ($q_{\hat{0}}$)")
    ax2.streamplot(x_vals, z_vals, Ex, Ez, color='black', density=1.5, linewidth=0.8)
    ax2.add_patch(plt.Circle((0, 0), horizon_radius, color='black', zorder=10))
    ax2.set_aspect('equal')
    ax2.set_title(r"Macroscopic Charge: $q_{\hat{0}}$", fontsize=14, fontweight='bold')

    plt.tight_layout()
    plt.show()

In [ ]:

#Corrected FLRW implementation
# 1. Define standard test values for physics parameters
m_val = 1.0   
q_val = 0.8   
a_val = 0.99 
qm_val = 0.8  

param_dict = {}
if M is not None: param_dict[M] = m_val
if Q is not None: param_dict[Q] = q_val
if a is not None: param_dict[a] = a_val
if Qm is not None: param_dict[Qm] = qm_val

# 2. FLRW COSMOLOGY: Substitute scale factor AND expansion rate
if syms.get('a_t') is not None:
    a_t_sym = syms['a_t']
    t_sym = syms['t']
    
    # Set the scale factor to 1.0 (Current size of universe)
    param_dict[a_t_sym] = 1.0
    
    # Set the time derivative (Hubble expansion rate H)
    # We use 0.5 for the visualizer to show a strong density field
    param_dict[sp.diff(a_t_sym, t_sym)] = 0.5 

# 3. Automatically route to the correct plotter
if M is None:
    print("✅ Cosmological metric detected. No horizon.")
    plot_analytical_fields_cartesian(E_hat, q_hat, rho_hat, x, y, z, param_dict, horizon_radius=None)

elif x is not None:
    # CARTESIAN COORDINATE METRICS
    if a is not None: horizon_r = m_val + np.sqrt(m_val**2 - a_val**2)
    elif Q is not None: horizon_r = 0.5 * np.sqrt(m_val**2 - q_val**2)
    else: horizon_r = m_val / 2.0
        
    print(f"✅ Cartesian coordinates detected. Rendering horizon at r = {horizon_r:.3f}")
    plot_analytical_fields_cartesian(E_hat, q_hat, rho_hat, x, y, z, param_dict, horizon_radius=horizon_r)
    
else:
    # SPHERICAL COORDINATE METRICS
    if a is not None: horizon_r = m_val + np.sqrt(m_val**2 - a_val**2)
    elif Q is not None: horizon_r = m_val + np.sqrt(m_val**2 - q_val**2)
    elif Qm is not None: horizon_r = 1.62 * m_val
    else: horizon_r = 2.0 * m_val

    print(f"✅ Spherical coordinates detected. Rendering horizon at r = {horizon_r:.3f}")
    plot_analytical_fields_spherical(E_hat, q_hat, rho_hat, r, theta, param_dict, horizon_radius=horizon_r)

In [ ]:
# 1. Define standard test values for physics parameters
m_val = 1.0  # Mass of the Black Hole
q_val = 0.3   # Electric Charge (Reissner-Nordstrom)
a_val = 0.99 # Spin Parameter (Kerr)
qm_val = 0.8  # Magnetic Monopole Charge (Bardeen)

# 2. Automatically build the substitution dictionary
param_dict = {M: m_val}
if Q is not None:
    param_dict[Q] = q_val
if a is not None:
    param_dict[a] = a_val
if Qm is not None:
    param_dict[Qm] = qm_val

# 3. Automatically determine the Horizon Radius and route to the correct plotter
if x is not None:
    # --- CARTESIAN COORDINATE METRICS ---
    if a is not None:
        # Kerr-Schild Cartesian (Horizon in Kerr-Schild matches Boyer-Lindquist r)
        horizon_r = m_val + np.sqrt(m_val**2 - a_val**2)
    elif Q is not None:
        # Reissner-Nordstrom Isotropic Cartesian (horizon naturally shifts to r_bar)
        horizon_r = 0.5 * np.sqrt(m_val**2 - q_val**2)
    else:
        # Schwarzschild Isotropic Cartesian
        horizon_r = m_val / 2.0
        
    print(f"✅ Cartesian coordinates detected. Rendering horizon at r = {horizon_r:.3f}")
    plot_analytical_fields_cartesian(E_hat, q_hat, rho_hat, x, y, z, param_dict, horizon_radius=horizon_r)
    
else:
    # --- SPHERICAL COORDINATE METRICS ---
    if a is not None:
        # Kerr Boyer-Lindquist / Quasi-Isotropic
        horizon_r = m_val + np.sqrt(m_val**2 - a_val**2)
    elif Q is not None:
        # Reissner-Nordstrom Spherical
        horizon_r = m_val + np.sqrt(m_val**2 - q_val**2)
    elif Qm is not None:
        # Bardeen Regular Black Hole (Approximate outer horizon)
        horizon_r = 1.62 * m_val
    else:
        # Schwarzschild Spherical
        horizon_r = 2.0 * m_val

    print(f"✅ Spherical coordinates detected. Rendering horizon at r = {horizon_r:.3f}")
    plot_analytical_fields_spherical(E_hat, q_hat, rho_hat, r, theta, param_dict, horizon_radius=horizon_r)

In [ ]:
def plot_analytical_fields(E_hat, q_hat, rho_hat, r_sym, theta_sym, M_sym, M_val=1.0):
    print("Lambdifying exact symbolic expressions into NumPy functions...")
    
    # Substitute Mass value into expressions
    E_r_expr = sp.simplify(E_hat[0, 1].subs(M_sym, M_val))
    E_theta_expr = sp.simplify(E_hat[0, 2].subs(M_sym, M_val))
    q0_expr = sp.simplify(q_hat[0].subs(M_sym, M_val))
    rho0_expr = sp.simplify(rho_hat[0].subs(M_sym, M_val))
    
    # Compile to fast C/NumPy functions
    E_r_func = lambdify((r_sym, theta_sym), E_r_expr, "numpy")
    E_theta_func = lambdify((r_sym, theta_sym), E_theta_expr, "numpy")
    q0_func = lambdify((r_sym, theta_sym), q0_expr, "numpy")
    rho0_func = lambdify((r_sym, theta_sym), rho0_expr, "numpy")
    
    print("Setting up grid and evaluating...")
    grid_lim = 8.0
    resolution = 300
    x_vals = np.linspace(-grid_lim, grid_lim, resolution)
    z_vals = np.linspace(-grid_lim, grid_lim, resolution)
    X, Z = np.meshgrid(x_vals, z_vals)
    
    R = np.sqrt(X**2 + Z**2)
    R = np.where(R < 1e-5, 1e-5, R) 
    Theta = np.arccos(Z / R)
    
    rs_val = 2.0 * M_val
    mask = R > (1.01 * rs_val)
    
    R_safe = np.where(mask, R, 1.01 * rs_val)
    Theta_safe = np.where(mask, Theta, np.pi/2)
    
    # Vectorized evaluation
    E_r_num = np.zeros_like(R)
    E_theta_num = np.zeros_like(R)
    q0_num = np.zeros_like(R)
    rho0_num = np.zeros_like(R)
    
    E_r_num[mask] = E_r_func(R_safe[mask], Theta_safe[mask])
    E_theta_num[mask] = E_theta_func(R_safe[mask], Theta_safe[mask])
    q0_num[mask] = q0_func(R_safe[mask], Theta_safe[mask])
    rho0_num[mask] = rho0_func(R_safe[mask], Theta_safe[mask])
    
    # Correct 3D to 2D Projection
    sign_x = np.sign(X)
    sign_x[sign_x == 0] = 1.0 
    
    Ex = (E_r_num * np.sin(Theta) + E_theta_num * np.cos(Theta)) * sign_x
    Ez = E_r_num * np.cos(Theta) - E_theta_num * np.sin(Theta)
    
    # Apply Horizon Mask
    Ex[~mask] = np.nan
    Ez[~mask] = np.nan
    q0_num[~mask] = np.nan
    rho0_num[~mask] = np.nan

    print("Rendering Plots...")
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    
    # Panel 1: Physical Density
    ax1 = axes[0]
    rho_max = np.nanmax(np.abs(rho0_num))
    if np.isnan(rho_max) or rho_max == 0: rho_max = 1.0
    cmap1 = ax1.pcolormesh(X, Z, rho0_num, shading='auto', cmap='RdBu_r', vmin=-rho_max, vmax=rho_max)
    fig.colorbar(cmap1, ax=ax1, label=r"Gravitational Energy Density ($\rho_{\hat{0}}$)")
    ax1.streamplot(x_vals, z_vals, Ex, Ez, color='black', density=1.5, linewidth=0.8, arrowsize=1.2)
    ax1.add_patch(plt.Circle((0, 0), rs_val, color='black', zorder=10))
    ax1.set_aspect('equal')
    ax1.set_title(r"True Energy Density: $\rho_{\hat{0}}$ (Spherically Symmetric)", fontsize=14, fontweight='bold')
    ax1.set_xlabel("X (M)", fontsize=12)
    ax1.set_ylabel("Z (M)", fontsize=12)

    # Panel 2: Macroscopic Charge
    ax2 = axes[1]
    q_max = np.nanmax(np.abs(q0_num))
    if np.isnan(q_max) or q_max == 0: q_max = 1.0 
    cmap2 = ax2.pcolormesh(X, Z, q0_num, shading='auto', cmap='RdBu_r', vmin=-q_max, vmax=q_max)
    fig.colorbar(cmap2, ax=ax2, label=r"Spacetime Energy Charge ($q_{\hat{0}}$)")
    ax2.streamplot(x_vals, z_vals, Ex, Ez, color='black', density=1.5, linewidth=0.8, arrowsize=1.2)
    ax2.add_patch(plt.Circle((0, 0), rs_val, color='black', zorder=10))
    ax2.set_aspect('equal')
    ax2.set_title(r"Macroscopic Charge: $q_{\hat{0}}$ (Volume Modulated)", fontsize=14, fontweight='bold')
    ax2.set_xlabel("X (M)", fontsize=12)

    plt.tight_layout()
    plt.show()

    # Pass the required SymPy objects into the plotting function
plot_analytical_fields(E_hat, q_hat, rho_hat, r, theta, M, M_val=1.0)

In [ ]:
def plot_analytical_fields_cartesian(E_hat, q_hat, rho_hat, x_sym, y_sym, z_sym, M_sym, M_val=1.0):
    print("Lambdifying exact symbolic expressions into NumPy functions...")
    
    # Extract Ex (index 1) and Ez (index 3) from the time-leg (index 0) of E_hat
    Ex_expr = sp.simplify(E_hat[0, 1].subs(M_sym, M_val))
    Ez_expr = sp.simplify(E_hat[0, 3].subs(M_sym, M_val))
    q0_expr = sp.simplify(q_hat[0].subs(M_sym, M_val))
    rho0_expr = sp.simplify(rho_hat[0].subs(M_sym, M_val))
    
    # Compile to fast C/NumPy functions using (x, y, z)
    Ex_func = lambdify((x_sym, y_sym, z_sym), Ex_expr, "numpy")
    Ez_func = lambdify((x_sym, y_sym, z_sym), Ez_expr, "numpy")
    q0_func = lambdify((x_sym, y_sym, z_sym), q0_expr, "numpy")
    rho0_func = lambdify((x_sym, y_sym, z_sym), rho0_expr, "numpy")
    
    print("Setting up grid and evaluating...")
    grid_lim = 8.0
    resolution = 300
    x_vals = np.linspace(-grid_lim, grid_lim, resolution)
    z_vals = np.linspace(-grid_lim, grid_lim, resolution)
    X, Z = np.meshgrid(x_vals, z_vals)
    Y = np.zeros_like(X) # Evaluate exactly on the equatorial slice y=0
    
    R = np.sqrt(X**2 + Z**2)
    
    # Mask out the interior of the black hole 
    # In isotropic coords, the horizon is at r_bar = M/2
    horizon_r_bar = M_val / 2.0
    mask = R > (1.05 * horizon_r_bar)
    
    # Safe arrays bypass math domain errors inside the horizon
    X_safe = np.where(mask, X, 1.05 * horizon_r_bar)
    Y_safe = np.where(mask, Y, 0.0)
    Z_safe = np.where(mask, Z, 0.0)
    
    # Vectorized evaluation directly using X, Y, Z
    Ex_num = np.zeros_like(R)
    Ez_num = np.zeros_like(R)
    q0_num = np.zeros_like(R)
    rho0_num = np.zeros_like(R)
    
    Ex_num[mask] = Ex_func(X_safe[mask], Y_safe[mask], Z_safe[mask])
    Ez_num[mask] = Ez_func(X_safe[mask], Y_safe[mask], Z_safe[mask])
    q0_num[mask] = q0_func(X_safe[mask], Y_safe[mask], Z_safe[mask])
    rho0_num[mask] = rho0_func(X_safe[mask], Y_safe[mask], Z_safe[mask])
    
    # Apply Horizon Mask
    Ex_num[~mask] = np.nan
    Ez_num[~mask] = np.nan
    q0_num[~mask] = np.nan
    rho0_num[~mask] = np.nan

    print("Rendering Plots...")
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    
    # Panel 1: Physical Density
    ax1 = axes[0]
    rho_max = np.nanmax(np.abs(rho0_num))
    if np.isnan(rho_max) or rho_max == 0: rho_max = 1.0
    cmap1 = ax1.pcolormesh(X, Z, rho0_num, shading='auto', cmap='RdBu_r', vmin=-rho_max, vmax=rho_max)
    fig.colorbar(cmap1, ax=ax1, label=r"Gravitational Energy Density ($\rho_{\hat{0}}$)")
    ax1.streamplot(x_vals, z_vals, Ex_num, Ez_num, color='black', density=1.5, linewidth=0.8)
    ax1.add_patch(plt.Circle((0, 0), horizon_r_bar, color='black', zorder=10))
    ax1.set_aspect('equal')
    ax1.set_title(r"True Energy Density: $\rho_{\hat{0}}$ (Cartesian Isotropic)", fontsize=14, fontweight='bold')
    ax1.set_xlabel("X (M)", fontsize=12)
    ax1.set_ylabel("Z (M)", fontsize=12)

    # Panel 2: Macroscopic Charge
    ax2 = axes[1]
    q_max = np.nanmax(np.abs(q0_num))
    if np.isnan(q_max) or q_max == 0: q_max = 1.0 
    cmap2 = ax2.pcolormesh(X, Z, q0_num, shading='auto', cmap='RdBu_r', vmin=-q_max, vmax=q_max)
    fig.colorbar(cmap2, ax=ax2, label=r"Spacetime Energy Charge ($q_{\hat{0}}$)")
    ax2.streamplot(x_vals, z_vals, Ex_num, Ez_num, color='black', density=1.5, linewidth=0.8)
    ax2.add_patch(plt.Circle((0, 0), horizon_r_bar, color='black', zorder=10))
    ax2.set_aspect('equal')
    ax2.set_title(r"Macroscopic Charge: $q_{\hat{0}}$", fontsize=14, fontweight='bold')
    ax2.set_xlabel("X (M)", fontsize=12)

    plt.tight_layout()
    plt.show()

# Pass the required SymPy objects into the plotting function
plot_analytical_fields_cartesian(E_hat, q_hat, rho_hat, x, y, z, M, M_val=3.0)

In [ ]:
# Pass the required SymPy objects into the plotting function
plot_analytical_fields_cartesian(E_hat, q_hat, rho_hat, x, y, z, M, M_val=3.0)

In [ ]:
# ==========================================
# MASTER TENSOR VISUALIZATION DASHBOARD
# ==========================================

# 1. Choose your Target Tensor and your Slice Plane!
TARGET_FIELD = D_hat     # Options: E_hat, B_hat, D_hat, H_hat, s_hat, j_hat
FIELD_NAME = "q"         # Text label for the plots (e.g., "E", "B", "s")
SLICE_PLANE = "Y"        # Options: "X", "Y", "Z"

# 2. Define physics parameters
m_val = 1.0   
q_val = 0.8   
a_val = 0.8   
qm_val = 0.8  

param_dict = {}
if M is not None: param_dict[M] = m_val
if Q is not None: param_dict[Q] = q_val
if a is not None: param_dict[a] = a_val
if Qm is not None: param_dict[Qm] = qm_val
if syms.get('a_t') is not None:
    param_dict[syms['a_t']] = 1.0
    param_dict[sp.diff(syms['a_t'], syms['t'])] = 0.5 

# 3. Automatically determine Horizon and Route to Slicer
if x is None:
    print("⚠️ Error: Slicing arbitrary planes (X=0, Y=0, Z=0) requires Cartesian coordinates.")
    print("Please load a Cartesian metric (e.g., get_kerr_schild_cartesian) in Cell 1.")
else:
    # Determine the correct horizon for Cartesian metrics
    if a is not None: horizon_r = m_val + np.sqrt(m_val**2 - a_val**2)
    elif Q is not None: horizon_r = 0.5 * np.sqrt(m_val**2 - q_val**2)
    elif M is None: horizon_r = None # Cosmological
    else: horizon_r = m_val / 2.0
        
    print(f"✅ Cartesian coordinates detected. Rendering horizon at r = {horizon_r:.3f}")
    plot_tensor_slice_cartesian(TARGET_FIELD, FIELD_NAME, SLICE_PLANE, x, y, z, param_dict, horizon_radius=horizon_r)